# Prompt & Agent Optimization with Opik — an A-to-Z Guide

This notebook takes you from *"I have a prompt that works okay"* to *"my prompt and my agent are measurably better, and every improvement is a comparable run in Opik."*

**Two ways to use it:**
- **Live workshop (≈20 min):** run **Part 1** top to bottom. You'll optimize a RAG answer prompt against an exact-match metric and see the improvement in Opik.
- **Take-home guide:** continue through Parts 2–5 — LLM-judge metrics (and how to *trust* them), multi-objective optimization, and optimizing an agent's tool use.

We optimize a **documentation assistant** for a fictional product, **Ledgerline** (a task-queue API), so the corpus is clean and the lesson is about *optimization*, not about parsing messy docs.

## Part 0 — How to think about prompt optimization

**When do you start?** When you have (1) a prompt that works *okay*, (2) a dataset of representative inputs, and (3) a metric that says how good an output is — and hand-tuning has plateaued.

**The mental shift.** Classic optimization gives you an objective and a gradient. Prompt optimization is different: the **search space is prompt text**, and the **objective is a metric computed over a dataset**. You can't differentiate it, so optimizers *propose* candidate prompts, *evaluate* them on your dataset, keep the best, and repeat.

The three ingredients map exactly to three objects you'll build:

| Ingredient | Opik object |
|---|---|
| The prompt | `ChatPrompt` |
| The dataset | Opik `Dataset` |
| The metric | a callable `(dataset_item, llm_output) -> float` |

The loop, once, looks like: **propose candidate → evaluate on dataset → keep best → repeat.** Everything below is that loop, escalating in complexity.

In [ ]:
import opik
from optimization_guide import config, data, rag_app

# Fail fast with a clear message if credentials are missing.
config.check_prerequisites()

client = opik.Opik(project_name=config.PROJECT_NAME)
print("Using models:", config.GEN_MODEL)

### Part 1 — Your first optimization ⭐ (workshop)

We'll ingest the Ledgerline docs, build an evaluation dataset with **checkable answers**, score a baseline prompt with an **exact-match metric** (no LLM judge needed), then let an optimizer improve the prompt.

In [ ]:
docs = data.load_docs()
count = rag_app.ingest(docs)
print(f"Ingested {count} doc snippets into ChromaDB")

In [ ]:
exact_cases = data.load_exact_cases()
exact_dataset = data.build_dataset(client, "ledgerline-exact", exact_cases)
print(f"Dataset 'ledgerline-exact' has {len(exact_cases)} cases")

#### The metric: exact-match, no judge

Our first metric is deterministic and cheap: **does the answer contain the expected fact?** Opik ships `Contains` for exactly this. Optimizer metrics are plain callables `(dataset_item, llm_output) -> float`, so we wrap `Contains` in one. *Not every metric needs an LLM.*

In [ ]:
from opik.evaluation.metrics import Contains


def exact_match(dataset_item: dict, llm_output: str) -> float:
    # Contains returns 1.0 if expected_substring is in the output, else 0.0.
    result = Contains(case_sensitive=False).score(
        output=llm_output,
        reference=dataset_item["expected_substring"],
    )
    return result.value


exact_match.__name__ = "exact_match"

#### The starting prompt

Here is our baseline system prompt — deliberately mediocre, so there's room to improve. This is the `ChatPrompt` the optimizer will rewrite. `{query}` is filled from each dataset row.

In [ ]:
from opik_optimizer import ChatPrompt

BASELINE_SYSTEM = "You are a support bot. Answer the question."

prompt = ChatPrompt(
    name="ledgerline-answer",
    system=BASELINE_SYSTEM,
    user="{query}",
    model=config.GEN_MODEL,
)

#### Run the optimizer

We use **`MetaPromptOptimizer`** — it uses a reasoning LLM to critique and rewrite the prompt. It's the docs' recommended general-purpose starting point for prompt wording. Watch the params:
- `max_trials` — how many candidate prompts to try.
- `n_samples` — dataset rows evaluated per candidate (smaller = cheaper/faster for a live run).
- `skip_perfect_score=False` — keep optimizing even if the baseline already scores high.

In [ ]:
from opik_optimizer import MetaPromptOptimizer

optimizer = MetaPromptOptimizer(
    model=config.OPTIMIZER_MODEL,
    n_threads=4,
    skip_perfect_score=False,
)

result = optimizer.optimize_prompt(
    prompt=prompt,
    dataset=exact_dataset,
    metric=exact_match,
    max_trials=8,
    n_samples=8,
)

print("Baseline score:", result.initial_score)
print("Best score:    ", result.score)
print("\nOptimized system prompt:\n", result.prompt)

#### See it in Opik

Open **Evaluation → Optimization runs** in your Opik workspace. You'll see this run with every candidate prompt, its score, and the trace for each trial. Compare the baseline row to the best row — that delta is your improvement.

🎓 **This is where the live workshop ends.** You've run a real optimization and improved a prompt, measured against a dataset, stored in Opik. Everything below builds on exactly this loop.